1️⃣ Imports & utilities

In [1]:
import numpy as np
from numpy.linalg import solve
from scipy.linalg import orthogonal_procrustes


2️⃣ Data simulation (OU latent factors)

In [2]:
def simulate_ou_dfa(
    n_subjects=500,
    T=4,
    p=7000,
    k=20,
    rho=0.5,
    Q=1.0,
    noise=0.3,
    seed=0,
):
    rng = np.random.default_rng(seed)

    # sparse loadings
    Lambda = rng.normal(0, 1, size=(p, k))
    mask = rng.random((p, k)) < 0.2
    Lambda *= mask

    Ys, Zs, dts = [], [], []

    for _ in range(n_subjects):
        z = np.zeros((T, k))
        dt = rng.uniform(0.8, 1.2, size=T-1)

        for t in range(1, T):
            a = np.exp(-rho * dt[t-1])
            z[t] = a * z[t-1] + rng.normal(0, np.sqrt(Q), size=k)

        y = z @ Lambda.T + rng.normal(0, noise, size=(T, p))

        Ys.append(y)
        Zs.append(z)
        dts.append(dt)

    return Ys, Zs, Lambda, dts


3️⃣ Kalman smoother (E-step)

Diagonal OU → independent dimensions → fast & stable.

In [3]:
def kalman_smoother(Y, Lambda, Psi, rho, Q, dt):
    T, p = Y.shape
    k = Lambda.shape[1]
    
    m = np.zeros(k)
    P = np.eye(k)
    
    Ez = np.zeros((T, k))
    Ezz = np.zeros((T, k, k))
    
    R = np.diag(Psi)
    eps = 1e-5
    
    Ez[0] = m
    Ezz[0] = P + np.outer(m, m)
    
    # Loop over intervals (t = 1..T-1)
    for t in range(1, T):
        # OU dynamics: use dt[t-1], NOT dt[t]
        A = np.exp(-rho * dt[t-1]) * np.eye(k)
        P = A @ P @ A.T + Q * np.eye(k)
        
        S = Lambda @ P @ Lambda.T + R + eps * np.eye(p)
        K = P @ Lambda.T @ np.linalg.solve(S, np.eye(p))
        
        m = A @ m + K @ (Y[t] - Lambda @ (A @ m))
        P = P - K @ Lambda @ P
        
        Ez[t] = m
        Ezz[t] = P + np.outer(m, m)
    
    # Lagged covariance
    Ezz_lag = np.zeros((T-1, k, k))
    for t in range(1, T):
        A = np.exp(-rho * dt[t-1]) * np.eye(k)
        Ezz_lag[t-1] = A @ P
    
    return Ez, Ezz, Ezz_lag


4️⃣ M-step updates
🔹 OU parameters ($ρ$ and $Q$)

In [4]:
def update_rho(Ezz, Ezz_lag, dts):
    num, den = 0.0, 0.0
    for i in range(len(Ezz)):
        for t in range(1, len(Ezz[i])):
            dt = dts[i][t-1]
            num += np.trace(Ezz_lag[i][t-1])
            den += dt * np.trace(Ezz[i][t-1])
    return max(num / den, 1e-3)


def update_Q(Ezz, Ezz_lag, rho, dts):
    num, den = 0.0, 0
    for i in range(len(Ezz)):
        for t in range(1, len(Ezz[i])):
            a = np.exp(-rho * dts[i][t-1])
            num += np.trace(Ezz[i][t] - a * Ezz_lag[i][t-1])
            den += Ezz[i][t].shape[0]
    return max(num / den, 1e-4)


🔹 Horseshoe-style $\Lambda$ update

In [5]:
def update_local_scales(Lambda, eps=1e-6):
    return 1.0 / (Lambda**2 + eps)


def update_global_scale(Lambda):
    return np.median(np.abs(Lambda))


def update_Lambda(Ez_all, Ezz_all, Y_all, Psi, tau, lam_k):
    """
    Update global loading matrix Lambda (p x k)

    Ez_all: (N*T, k)
    Ezz_all: (N*T, k, k)
    Y_all: (N*T, p)
    Psi: (p,)
    tau: scalar
    lam_k: (k,)  column-wise shrinkage
    """
    NT, k = Ez_all.shape
    p = Y_all.shape[1]

    # Left term: sum y zᵀ
    S_yz = Y_all.T @ Ez_all   # (p, k)

    # Right term: sum E[zzᵀ]
    S_zz = np.sum(Ezz_all, axis=0)  # (k, k)

    # Horseshoe-style ridge approximation
    prior = np.diag(1.0 / (tau**2 * lam_k**2 + 1e-8))

    # Solve for Lambda row-wise (scales to p=10k)
    A = S_zz + prior
    A_inv = np.linalg.inv(A)

    Lambda = S_yz @ A_inv     # (p, k)

    # 🚨 HARD SAFETY CHECK
    assert Lambda.ndim == 2

    return Lambda



🔹 Observation noise

In [6]:
def update_Psi(Ys, Ezs, Ezzs, Lambda):
    """
    Diagonal observation noise update.
    Psi is p-dimensional (diagonal covariance).
    """
    p, k = Lambda.shape
    num = np.zeros(p)
    den = 0

    for Y, Ez, Ezz in zip(Ys, Ezs, Ezzs):
        T = Y.shape[0]
        for t in range(T):
            # Ez[t] must be (k,)
            assert Ez[t].shape == (k,)
            assert Ezz[t].shape == (k, k)

            y = Y[t]                          # (p,)
            mu = Lambda @ Ez[t]               # (p,)

            # E[(y - Λz)^2] = (y - ΛEz)^2 + diag(Λ Var(z) Λᵀ)
            resid2 = (y - mu) ** 2
            var_term = np.sum(
                (Lambda @ Ezz[t]) * Lambda, axis=1
            )

            num += resid2 + var_term
            den += 1

    return num / den


5️⃣ Rotation-invariant recovery metric

In [7]:
def factor_recovery(L_true, L_est):
    R, _ = orthogonal_procrustes(L_est, L_true)
    L_aligned = L_est @ R

    C = np.abs(np.corrcoef(
        L_true.T, L_aligned.T
    )[:L_true.shape[1], L_true.shape[1]:])

    return np.mean(np.max(C, axis=1))


6️⃣ Full EM loop (THIS is the fix)

In [8]:
def run_em(Ys, dts, k, n_iter=30):
    p = Ys[0].shape[1]

    Lambda = np.random.normal(0, 0.1, size=(p, k))
    Psi = np.eye(p)
    rho = 0.3
    Q = 1.0

    for it in range(n_iter):
        Ezs, Ezzs, Ezz_lags = [], [], []

        # E-step
        for Y, dt in zip(Ys, dts):
            Ez, Ezz, Ezz_lag = kalman_smoother(
                Y, Lambda, Psi, rho, Q, dt
            )
            Ezs.append(Ez)
            Ezzs.append(Ezz)
            Ezz_lags.append(Ezz_lag)

        # M-step: OU
        rho = update_rho(Ezzs, Ezz_lags, dts)
        Q = update_Q(Ezzs, Ezz_lags, rho, dts)

        # M-step: loadings
        lam = update_local_scales(Lambda)
        tau = update_global_scale(Lambda)

        Ez_all = np.vstack(Ezs)
        Ezz_all = sum(Ezzs)
        Y_all = np.vstack(Ys)

        Lambda = update_Lambda(
            Ez_all, Ezz_all, Y_all, Psi, tau, lam.mean(axis=0)
        )
        assert Lambda.shape == (p, k)

        Psi = update_Psi(Ys, Ezs, Ezzs, Lambda)

        print(
            f"Iter {it:02d} | rho={rho:.3f} | Q={Q:.3f} | "
            f"mean|Λ|={np.mean(np.abs(Lambda)):.3f}"
        )

    return Lambda


7️⃣ Main function (run & validate)

In [9]:
def main():
    Ys, Zs, Lambda_true, dts = simulate_ou_dfa(
        n_subjects=60,
        T=4,
        p=500,
        k=20,
        rho=0.5,
        Q=1.0,
    )

    Lambda_est = run_em(Ys, dts, k=20, n_iter=30)

    rec = factor_recovery(Lambda_true, Lambda_est)
    print("\nProcrustes-aligned factor recovery:", rec)


if __name__ == "__main__":
    main()


Iter 00 | rho=0.001 | Q=0.878 | mean|Λ|=0.250
Iter 01 | rho=0.071 | Q=3.203 | mean|Λ|=0.213
Iter 02 | rho=0.050 | Q=3.975 | mean|Λ|=0.187
Iter 03 | rho=0.051 | Q=4.616 | mean|Λ|=0.170
Iter 04 | rho=0.051 | Q=5.302 | mean|Λ|=0.156
Iter 05 | rho=0.052 | Q=6.021 | mean|Λ|=0.146
Iter 06 | rho=0.052 | Q=6.767 | mean|Λ|=0.137
Iter 07 | rho=0.053 | Q=7.537 | mean|Λ|=0.129
Iter 08 | rho=0.053 | Q=8.330 | mean|Λ|=0.123
Iter 09 | rho=0.053 | Q=9.144 | mean|Λ|=0.117
Iter 10 | rho=0.054 | Q=9.978 | mean|Λ|=0.113
Iter 11 | rho=0.054 | Q=10.833 | mean|Λ|=0.108
Iter 12 | rho=0.054 | Q=11.708 | mean|Λ|=0.104
Iter 13 | rho=0.054 | Q=12.604 | mean|Λ|=0.100
Iter 14 | rho=0.055 | Q=13.520 | mean|Λ|=0.097
Iter 15 | rho=0.055 | Q=14.458 | mean|Λ|=0.094
Iter 16 | rho=0.055 | Q=15.416 | mean|Λ|=0.091
Iter 17 | rho=0.055 | Q=16.397 | mean|Λ|=0.088
Iter 18 | rho=0.055 | Q=17.399 | mean|Λ|=0.086
Iter 19 | rho=0.055 | Q=18.423 | mean|Λ|=0.084
Iter 20 | rho=0.055 | Q=19.470 | mean|Λ|=0.081
Iter 21 | rho=0.056 | Q=